# Equipo 2

- Acevedo Juárez Sebastián
- Madariaga Villanueva Fernanda
- Pérez Castillo José Eduardo
- Rodríguez Cruz Derek
- Trejo Salinas Jared

In [39]:
# instalacion
# %pip install faster-whisper sounddevice soundfile piper-tts transformers torch numpy gensim

In [40]:
import numpy as np
import re
from datetime import datetime
from pathlib import Path
from gensim.models import Word2Vec

In [41]:
# log
def log(msg, tipo="info"):
    etq = {"info":"        ", 
           "tu":"  [TU]   ", 
           "alexa":"  [ALEXA]",
           "det":"  [DETEC]", 
           "estado":">>ESTADO"}
    print(etq.get(tipo, "        ") + "  " + str(msg))

In [42]:
# preprocesamiento
def A_minusculas(texto):
    salida = ""
    for c in texto:
        o = ord(c)
        if o >= 65 and o <= 90:
            c = chr(o + 32)
        salida += c
    return salida

LETRAS = "abcdefghijklmnopqrstuvwxyzñáéíóúü"
def tokenizar(texto):
    texto = A_minusculas(texto)
    tokens = []
    actual = ""
    for c in texto:
        if c in LETRAS:
            actual += c
        else:
            if actual != "":
                tokens += [actual]
                actual = ""
    if actual != "":
        tokens += [actual]
    return tokens

In [43]:
# primitivas: coseno, ngramas, tfidf
def coseno(a, b):
    # sacamos norma y producto punto
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def perfil_ngramas(texto, n=2):
    texto = A_minusculas(texto)
    perfil = {}
    i = 0
    while i <= len(texto) - n:
        g = texto[i:i+n]
        if g in perfil:
            perfil[g] += 1
        else:
            perfil[g] = 1
        i += 1
    return perfil

def coseno_perfiles(p1, p2):
    claves = {}

    for k in p1: claves[k] = True
    for k in p2: claves[k] = True

    dot = 0.0
    s1 = 0.0
    s2 = 0.0

    for k in claves:
        v1 = p1[k] if k in p1 else 0
        v2 = p2[k] if k in p2 else 0

        # producto punto
        dot += v1 * v2

        # Cuadrado de sus componentes
        s1 += v1 * v1
        s2 += v2 * v2

    if s1 == 0 or s2 == 0:
        return 0.0
    
    return dot / ((s1 ** 0.5) * (s2 ** 0.5))

def _vocabulario(corpus_tokens):
    vocab = []
    for toks in corpus_tokens:
        for t in toks:
            if t not in vocab:
                vocab += [t]
    return vocab

def tfidf_fit(corpus_tokens):
    vocab = _vocabulario(corpus_tokens)
    N = len(corpus_tokens)
    idf = {}
    for t in vocab:
        df = 0
        for toks in corpus_tokens:
            if t in toks:
                df += 1
        idf[t] = np.log((N + 1) / (df + 1)) + 1.0
    return vocab, idf

def tfidf_vector(tokens, vocab, idf):
    v = np.zeros(len(vocab))
    total = len(tokens)
    if total == 0:
        return v
    j = 0
    while j < len(vocab):
        t = vocab[j]
        c = 0
        for w in tokens:
            if w == t:
                c += 1
        v[j] = (c / total) * idf[t]
        j += 1
    return v

In [44]:
p = perfil_ngramas("alexa que cuentas", 2)
print(p)

s = coseno_perfiles(p, perfil_ngramas("alexa que onda", 2))

print(s)

{'al': 1, 'le': 1, 'ex': 1, 'xa': 1, 'a ': 1, ' q': 1, 'qu': 1, 'ue': 2, 'e ': 1, ' c': 1, 'cu': 1, 'en': 1, 'nt': 1, 'ta': 1, 'as': 1}
0.6537204504606136


In [45]:
# reconocer integrante (ngramas)
INTEGRANTES = {
    "Edu":  "hola alexa",
    "Fer":  "alexa que onda",
    "Sebas": "alexa que cuentas",
    "Derek": "alexa como estas",
    "Jared": "alexa que hay",
}

UMBRAL_SALUDO = 0.55

def reconocer_integrante(texto_dicho):
    p = perfil_ngramas(texto_dicho, 2)
    mejor = None
    mejor_s = -1.0
    
    for nombre in INTEGRANTES:
        s = coseno_perfiles(p, perfil_ngramas(INTEGRANTES[nombre], 2))
        if s > mejor_s:
            mejor_s = s
            mejor = nombre
    if mejor_s >= UMBRAL_SALUDO:
        return mejor, mejor_s
    return None, mejor_s

In [46]:
# sentimiento (bert)
_BERT = {}
def analizar_sentimiento(texto):
    if "pipe" not in _BERT:
        from transformers import pipeline
        _BERT["pipe"] = pipeline(
            "sentiment-analysis",
            model="pysentimiento/robertuito-sentiment-analysis")
    r = _BERT["pipe"](texto)[0]
    etiqueta = r["label"].upper()
    if etiqueta in ("POS", "POSITIVE"): return "positivo"
    if etiqueta in ("NEG", "NEGATIVE"): return "negativo"
    return "neutral"

In [47]:
# datos curiosos
DATOS_ANIO = {
    1975: "en 1975 se fundó Microsoft por Bill Gates y Paul Allen.",
    1976: "en 1976 se fundó Apple Computer en el garaje de Steve Jobs.",
    1977: "en 1977 se estrenó la primera película de Star Wars.",
    1978: "en 1978 nació el primer bebé concebido mediante fertilización in vitro.",
    1979: "en 1979 Sony revolucionó la música portátil al lanzar el primer Walkman.",
    1980: "en 1980 se lanzó al mercado el famoso videojuego arcade Pac-Man.",
    1981: "en 1981 IBM lanzó su primera computadora personal, popularizando las PC.",
    1982: "en 1982 salió a la venta Thriller de Michael Jackson, el álbum más vendido.",
    1983: "en 1983 nació oficialmente internet al adoptarse el protocolo TCP/IP.",
    1984: "en 1984 Apple presentó la computadora Macintosh con un famoso comercial.",
    1985: "en 1985 se lanzó la consola Nintendo Entertainment System (NES) en América.",
    1986: "en 1986 ocurrió el accidente nuclear en la planta de Chernóbil.",
    1987: "en 1987 la población mundial alcanzó la cifra de 5 mil millones de habitantes.",
    1988: "en 1988 se introdujo el primer cable de fibra óptica transatlántico.",
    1989: "en 1989 cayó el Muro de Berlín, comenzando la reunificación de Alemania.",
    1990: "en 1990 se lanzó al espacio el telescopio espacial Hubble.",
    1991: "en 1991 se publicó la primera página web de la historia por Tim Berners-Lee.",
    1992: "en 1992 se envió el primer mensaje de texto SMS a un teléfono móvil.",
    1993: "en 1993 se estrenó Jurassic Park, revolucionando los efectos especiales CGI.",
    1994: "en 1994 se fundó Amazon, inicialmente como una librería en internet.",
    1995: "en 1995 se lanzó el sistema operativo Windows 95 y se inventó el formato DVD.",
    1996: "en 1996 la oveja Dolly se convirtió en el primer mamífero clonado.",
    1997: "en 1997 la supercomputadora Deep Blue derrotó al campeón mundial de ajedrez.",
    1998: "en 1998 se fundó Google en un pequeño garaje de California.",
    1999: "en 1999 el mundo tecnológico se preparó en pánico para el error informático Y2K.",
    2000: "en el año 2000 la Estación Espacial Internacional recibió a sus primeros residentes.",
    2001: "en 2001 se lanzó Wikipedia, la enciclopedia libre y colaborativa.",
    2002: "en 2002 el Euro entró en circulación física en varios países de Europa.",
    2003: "en 2003 se completó el mapa del Proyecto del Genoma Humano.",
    2004: "en 2004 se fundó la red social Facebook en la Universidad de Harvard.",
    2005: "en 2005 se subió el primer video de la historia a la plataforma YouTube.",
    2006: "en 2006 Plutón perdió su estatus y fue reclasificado como planeta enano.",
    2007: "en 2007 se presentó al mundo el primer modelo de teléfono inteligente iPhone.",
    2008: "en 2008 se lanzó la primera versión comercial del sistema operativo móvil Android.",
    2009: "en 2009 se minó el bloque génesis de la primera criptomoneda, Bitcoin.",
    2010: "en 2010 se lanzó la aplicación Instagram, enfocada en compartir fotografías.",
    2011: "en 2011 finalizó oficialmente el programa de transbordadores espaciales de la NASA.",
    2012: "en 2012 la sonda Curiosity aterrizó de manera exitosa en la superficie de Marte.",
    2013: "en 2013 comenzó una nueva generación de videojuegos con la PS4 y la Xbox One.",
    2014: "en 2014 la humanidad logró aterrizar una sonda espacial en un cometa.",
    2015: "en 2015 decenas de países firmaron el Acuerdo de París sobre cambio climático.",
    2016: "en 2016 el juego de realidad aumentada Pokémon GO causó un fenómeno mundial.",
    2017: "en 2017 se descubrió Oumuamua, el primer objeto interestelar conocido.",
    2018: "en 2018 un cohete Falcon Heavy fue lanzado al espacio llevando un automóvil rojo.",
    2019: "en 2019 científicos lograron capturar la primera imagen fotográfica de un agujero negro.",
    2020: "en 2020 el mundo vivió una pandemia global que popularizó el trabajo remoto.",
    2021: "en 2021 se lanzó al espacio el avanzado telescopio espacial James Webb.",
    2022: "en 2022 la población humana mundial superó los 8 mil millones de habitantes.",
    2023: "en 2023 la inteligencia artificial generativa se volvió accesible para el público general.",
    2024: "en 2024 la ciudad de París albergó los Juegos Olímpicos de verano.",
    2025: "en 2025 se lanzó la esperada versión 3 de Apache Airflow a principios de año."
}

def dato_anio(anio):
    if anio in DATOS_ANIO:
        return DATOS_ANIO[anio]
    
    mejor = None
    dist = None
    
    for a in DATOS_ANIO:
        d = abs(a - anio)
        if dist is None or d < dist:
            dist = d
            mejor = a
    if mejor is None:
        return "no tengo un dato para ese año."
    return DATOS_ANIO[mejor]

DATOS_PASATIEMPO = [
    {"clave":"futbol deporte balon cancha", "dato":"el futbol es el deporte mas visto del planeta, con miles de millones de seguidores."},
    {"clave":"musica tocar guitarra cantar", "dato":"escuchar musica libera dopamina en el cerebro, generando la misma respuesta que comer tu comida favorita."},
    {"clave":"videojuegos jugar consola xbox halo", "dato":"el primer juego de la franquicia Halo revolucionó los juegos de disparos en primera persona para consolas con su sistema de escudo regenerativo."},
    {"clave":"leer libros lectura novela", "dato":"leer apenas seis minutos al dia puede reducir los niveles de estres hasta en un 68 por ciento."},
    {"clave":"cocinar cocina recetas comida", "dato":"cocinar en casa activa zonas del cerebro fuertemente ligadas a la creatividad y la atencion plena."},
    {"clave":"correr running ejercicio gym", "dato":"correr de forma constante libera endorfinas, lo que provoca la famosa y adictiva euforia del corredor."},
    {"clave":"pintar dibujar arte", "dato":"dibujar conceptos a mano mejora la memoria mucho mas que simplemente escribir notas de texto."},
    {"clave":"motos motocicleta mecanica rodar ft150", "dato":"los sistemas de cambio rápido (quick shifters) permiten a los motociclistas subir de marcha sin usar el embrague, manteniendo el acelerador abierto."},
    {"clave":"boxeo box pelear gimnasio", "dato":"el boxeo no solo mejora dramáticamente la resistencia cardiovascular, sino que también entrena la coordinación ojo-mano a niveles de élite."},
    {"clave":"baile bailar salsa bachata reggaeton", "dato":"aprender a bailar coreografías de diferentes géneros mejora la neuroplasticidad del cerebro al forzar la coordinación rítmica y motriz."},
    {"clave":"oratoria hablar publico discursos toastmasters", "dato":"la glosofobia, o el miedo paralizante a hablar frente a una audiencia, llega a afectar a casi el 75 por ciento de la población mundial."},
    {"clave":"programacion arduino electronica microcontroladores avr", "dato":"manipular directamente los registros de memoria de un microcontrolador es muchísimo más eficiente y rápido que usar librerías de alto nivel."},
    {"clave":"datos bases airflow ingenieria", "dato":"las herramientas modernas de orquestación de datos aseguran que miles de tareas en sistemas complejos se ejecuten en el orden exacto sin dependencias rotas."},
    {"clave":"ia inteligencia artificial modelos locales", "dato":"ejecutar modelos de inteligencia artificial de forma local con aceleración por GPU puede reducir el tiempo de respuesta de minutos a solo segundos."},
    {"clave":"edicion video capcut inshot", "dato":"la edición de video en la actualidad permite aplicar cortes invisibles y efectos visuales complejos directamente desde un teléfono móvil."},
    {"clave":"ajedrez tablero piezas jaque", "dato":"matemáticamente, existen más partidas de ajedrez posibles que la cantidad de átomos en el universo observable."},
    {"clave":"natacion nadar alberca piscina", "dato":"la natación es uno de los pocos deportes que involucra el uso activo de casi todos los grupos musculares del cuerpo simultáneamente."},
    {"clave":"basquetbol baloncesto canasta", "dato":"el basquetbol se inventó en invierno con cestas de duraznos reales, por lo que inicialmente tenían que empujar el balón con un palo tras cada punto."},
    {"clave":"tenis raqueta red", "dato":"las pelotas de tenis son de color amarillo óptico porque en los primeros televisores a color ese tono era el más fácil de seguir en pantalla."},
    {"clave":"yoga meditacion posturas asanas", "dato":"la palabra yoga proviene del sánscrito y significa literalmente unión o esfuerzo."},
    {"clave":"ciclismo bicicleta pedalear", "dato":"puedes estacionar hasta quince bicicletas en el mismo espacio que ocupa un solo automóvil estándar."},
    {"clave":"patinaje patines hielo ruedas", "dato":"el primer par de patines sobre ruedas se inventó en 1760 y su creador se estrelló en un espejo gigante durante la presentación por no saber frenar."},
    {"clave":"fotografia fotos camara lentes", "dato":"la primera fotografía de la historia, tomada en 1826, requirió una exposición ininterrumpida de ocho horas al sol."},
    {"clave":"escultura arcilla barro esculpir", "dato":"miguel ángel creía que la escultura ya existía dentro del bloque de mármol y su único trabajo era quitar el material sobrante."},
    {"clave":"tejer costura hilo agujas", "dato":"tejer estimula áreas cerebrales similares a la meditación, reduciendo la presión arterial y la frecuencia cardíaca."},
    {"clave":"carpinteria madera muebles construir", "dato":"la carpintería sin clavos japonesa utiliza ensambles geométricos tan perfectos que pueden resistir terremotos sin pegamento."},
    {"clave":"jardineria plantas cultivar huerto", "dato":"trabajar con la tierra del jardín expone a bacterias amigables que pueden actuar como antidepresivos naturales en el cerebro humano."},
    {"clave":"reposteria hornear pasteles dulces", "dato":"la repostería requiere química exacta; un ligero cambio en la temperatura de la mantequilla puede cambiar por completo la textura de una galleta."},
    {"clave":"magia trucos ilusionismo cartas", "dato":"el cerebro humano tiene puntos ciegos de atención que los ilusionistas explotan científicamente para hacer magia de cerca."},
    {"clave":"teatro actuacion escenario", "dato":"en el teatro antiguo, las máscaras no solo representaban emociones, sino que funcionaban como megáfonos acústicos para proyectar la voz."},
    {"clave":"cine peliculas grabar cortometrajes", "dato":"las claquetas de cine tienen ese diseño de rayas blancas y negras para poder alinear el sonido y la imagen fácilmente al editar."},
    {"clave":"rompecabezas puzzles armar", "dato":"resolver rompecabezas refuerza las conexiones entre las células cerebrales y mejora la memoria a corto plazo de forma notable."},
    {"clave":"coleccionismo monedas figuras estampillas", "dato":"la numismática, o el coleccionismo de monedas, es conocido como el pasatiempo de los reyes debido a su origen en la realeza europea."},
    {"clave":"astronomia estrellas telescopio espacio", "dato":"si observas la estrella sirio en la noche, estás viendo la luz que emitió hace más de ocho años debido a la distancia."},
    {"clave":"impresion 3d pla filamento", "dato":"la tecnología de impresión 3d ya se utiliza para imprimir piezas biológicas y refacciones mecánicas directamente en la estación espacial internacional."},
    {"clave":"drones volar cuadricoptero", "dato":"los drones modernos pueden utilizar inteligencia artificial para seguir a un objetivo y evadir obstáculos físicos de forma autónoma."},
    {"clave":"billar mesas tacos bolas", "dato":"las bolas de billar originales estaban hechas de marfil de elefante puro, hasta que se inventaron los plásticos para evitar su extinción."},
    {"clave":"ping pong tenis mesa", "dato":"a nivel profesional, la pelota de ping pong puede llegar a viajar a velocidades superiores a los 110 kilómetros por hora."},
    {"clave":"atletismo saltos carreras pista", "dato":"los corredores de velocidad extrema pueden tener un impacto en el suelo equivalente a tres veces su peso corporal en cada zancada."},
    {"clave":"parkour saltar obstaculos urbano", "dato":"el parkour se originó en francia a partir de programas de entrenamiento de evasión militar para cruzar terrenos de forma rápida."},
    {"clave":"dj mezclar musica tornamesas", "dato":"el scratch, la famosa técnica de mover el disco de vinilo hacia atrás, fue inventada por accidente en 1975 por un joven dj en nueva york."},
    {"clave":"podcasting grabar audio microfono", "dato":"la palabra podcast es la combinación del reproductor ipod y la palabra broadcast, que significa transmisión."},
    {"clave":"diseño grafico ilustracion vectores", "dato":"la tipografía comic sans se creó originalmente solo para el texto de ayuda de un perrito virtual en un software de los años noventa."},
    {"clave":"animacion 2d 3d cuadros fps", "dato":"para crear solo un segundo de animación tradicional de alta calidad se requieren dibujar veinticuatro cuadros individuales distintos."},
    {"clave":"ciberseguridad hacking redes", "dato":"la primera contraseña de computadora de la historia fue inventada en 1961 para evitar que los estudiantes usaran demasiadas horas de un servidor central."},
    {"clave":"idiomas aprender lenguas poliglotas", "dato":"aprender un segundo idioma cambia físicamente tu cerebro, aumentando la densidad de la materia gris en las áreas del lenguaje."},
    {"clave":"buceo submarinismo mar tanque", "dato":"bajo el agua, los objetos parecen un veinticinco por ciento más grandes y más cercanos de lo que realmente están por la refracción de la luz."},
    {"clave":"pesca caña anzuelo lago", "dato":"los pescadores deportivos a menudo utilizan el principio de captura y liberación para ayudar a mantener el ecosistema de los cuerpos de agua."},
    {"clave":"senderismo caminar montaña cerro", "dato":"caminar por la naturaleza o senderismo reduce la rumiación, que es el patrón de pensamiento negativo asociado a la ansiedad."},
    {"clave":"origami papel papiroflexia doblar", "dato":"en japón, la grulla de papel es un símbolo de paz, y la leyenda dice que quien doble mil grullas verá cumplido su mayor deseo."}
]

DATOS_GUSTO = [
    {"clave":"tacos pozole comida mexicana", "dato":"el pozole tiene sus raíces en la época prehispánica y su nombre proviene de una palabra náhuatl que significa espumoso."},
    {"clave":"pizza pepperoni rebanada", "dato":"la pizza de pepperoni es la más consumida en gran parte del mundo, aunque el pepperoni en sí es un embutido inventado en estados unidos, no en italia."},
    {"clave":"cafe frappe starbucks bebida", "dato":"la versión moderna del café frappé fue inventada por pura casualidad en grecia, al no encontrar agua caliente para mezclar el café soluble."},
    {"clave":"chocolate dulce postre", "dato":"el chocolate no siempre fue un dulce; las civilizaciones prehispánicas lo tomaban como una bebida amarga, picante y lo usaban como moneda."},
    {"clave":"perros mascotas caninos", "dato":"los perros tienen la capacidad de entender el tono de voz humano e incluso pueden aprender el significado de más de 150 palabras diferentes."},
    {"clave":"gatos felinos mascotas ronroneo", "dato":"los gatos no tienen clavículas, lo que les permite deslizar su cuerpo por cualquier abertura por la que pueda pasar su cabeza."},
    {"clave":"cine peliculas series", "dato":"la primera película grabada en la historia duraba poco más de dos segundos y mostraba simplemente a unas personas caminando por un jardín."},
    {"clave":"viajar viajes avion turismo", "dato":"viajar frecuentemente y enfrentarse a nuevas culturas genera conexiones neuronales únicas que mantienen el cerebro biológicamente joven y ágil."},
    {"clave":"tecnologia computadoras gadgets", "dato":"la computadora que guió al hombre a la luna tenía mucha menos memoria ram que el chip de un microondas moderno."},
    {"clave":"naturaleza plantas bosque", "dato":"las plantas en un bosque están interconectadas por redes de hongos subterráneas a través de las cuales se envían nutrientes y alertas de peligro."},
    {"clave":"hamburguesa queso papas", "dato":"se estima que en todo el mundo se venden cientos de hamburguesas por segundo, siendo el alimento de comida rápida por excelencia."},
    {"clave":"sushi arroz pescado japones", "dato":"originalmente, el sushi no era un platillo de lujo, sino un método asiático para conservar pescado fermentándolo envuelto en arroz que luego se desechaba."},
    {"clave":"pasta espagueti italia", "dato":"existen más de seiscientos tipos diferentes de pasta en el mundo, y sus formas están diseñadas matemáticamente para atrapar mejor distintos tipos de salsas."},
    {"clave":"helado nieve postre frio", "dato":"el dolor de cabeza por comer helado muy rápido ocurre porque los nervios del paladar se enfrían y envían una señal de emergencia al cerebro."},
    {"clave":"vino uvas copa bebida", "dato":"para producir una sola botella estándar de vino de buena calidad, se necesitan aproximadamente un kilo y medio de uvas frescas."},
    {"clave":"cerveza cebada lúpulo", "dato":"la cerveza es una de las recetas humanas más antiguas; se han encontrado registros escritos de su preparación en antiguas tablas sumerias."},
    {"clave":"te matcha infusion verde", "dato":"todo el té del mundo, ya sea negro, verde o blanco, proviene exactamente de la misma especie de planta; la diferencia está en cómo se procesan las hojas."},
    {"clave":"manzana fruta saludable", "dato":"existen tantas variedades de manzanas cultivadas en el mundo que, si comieras una distinta al día, tardarías más de 20 años en probarlas todas."},
    {"clave":"mariscos camaron langosta", "dato":"las langostas tienen dientes ubicados dentro de sus estómagos y detectan la comida saboreándola directamente con sus patas al caminar."},
    {"clave":"picante chiles salsa", "dato":"lo que percibimos como sabor picante no es un gusto, sino una reacción de dolor químico detectada por los sensores de temperatura en nuestra lengua."},
    {"clave":"pajaros aves volar", "dato":"los colibríes son las únicas aves en el planeta Tierra que pueden volar hacia atrás y mantenerse suspendidos inmóviles en el aire."},
    {"clave":"peces acuario mar pez", "dato":"la mayoría de los peces no pueden nadar hacia atrás, y muchas especies necesitan mantenerse en movimiento constante simplemente para poder respirar oxígeno."},
    {"clave":"caballos equinos galope", "dato":"los caballos tienen los ojos más grandes de cualquier mamífero terrestre y pueden dormir tanto de pie como recostados debido a un mecanismo de bloqueo en sus patas."},
    {"clave":"dinosaurios t-rex prehistoria", "dato":"cronológicamente, el tiranosaurio rex vivió en una época más cercana al lanzamiento del primer iphone que a la existencia del estegosaurio."},
    {"clave":"extraterrestres ovnis espacio aliens", "dato":"la ecuación de drake estima que podrían existir miles de civilizaciones avanzadas comunicándose en nuestra galaxia, pero el universo es tan grande que no las oímos."},
    {"clave":"zombies muertos vivientes apocalipsis", "dato":"la palabra zombie proviene originalmente del folclore haitiano, asociado a prácticas reales de personas puestas en trance profundo con toxinas naturales."},
    {"clave":"magia fantasia libros hechiceria", "dato":"el concepto de la varita mágica tiene sus raíces en los textos griegos y romanos antiguos, donde los dioses la usaban para canalizar energía espiritual."},
    {"clave":"ciencia ficcion espacio naves", "dato":"la ciencia ficción ha inspirado tecnología real; el creador del primer teléfono móvil se inspiró en los comunicadores de una popular serie espacial de los años sesenta."},
    {"clave":"terror miedo sustos", "dato":"ver películas de terror controladas puede aumentar las proteínas glóbulos blancos circulantes, lo que curiosamente mejora la respuesta inmunológica temporalmente."},
    {"clave":"comedia risa chistes standup", "dato":"reír a carcajadas durante quince minutos quema aproximadamente la misma cantidad de calorías que realizar una caminata ligera continua."},
    {"clave":"anime manga animacion japonesa", "dato":"el anime representa más del sesenta por ciento del mercado mundial de entretenimiento animado basado en televisión."},
    {"clave":"superheroes comics poderes", "dato":"el primer cómic protagonizado por superman se vendió por diez centavos en 1938 y hoy en día una copia original en buen estado vale millones de dólares."},
    {"clave":"historia pasado civilizaciones", "dato":"cleopatra vivió cronológicamente en un punto del tiempo más cercano a la invención del ipad que a la construcción de las grandes pirámides de egipto."},
    {"clave":"ciencia fisica quimica", "dato":"en este momento exacto, miles de millones de diminutas partículas llamadas neutrinos provenientes del sol están atravesando tu cuerpo cada segundo sin que lo sientas."},
    {"clave":"arquitectura edificios construir diseño", "dato":"la torre eiffel puede llegar a ser quince centímetros más alta durante el verano debido a que el hierro se expande con el intenso calor del sol."},
    {"clave":"moda ropa estilo vestir", "dato":"los botones en las mangas de las chaquetas y sacos fueron inventados por orden militar para evitar que los soldados se limpiaran la nariz en los uniformes."},
    {"clave":"carros autos automoviles vehiculos", "dato":"un automóvil moderno promedio está compuesto por alrededor de treinta mil piezas individuales trabajando en conjunto de forma sincronizada."},
    {"clave":"aviones volar aeropuerto", "dato":"el aire dentro de la cabina de un avión en pleno vuelo a gran altitud suele ser más seco que el aire de uno de los desiertos más cálidos del mundo."},
    {"clave":"trenes ferrocarril vias", "dato":"el tren de levitación magnética más rápido del mundo puede viajar a velocidades cercanas a los 600 kilómetros por hora flotando sobre las vías."},
    {"clave":"relojes tiempo cronografo", "dato":"en los anuncios y catálogos mundiales, las manecillas de los relojes casi siempre se colocan a las 10:10 porque simula una sutil sonrisa de victoria."},
    {"clave":"joyas diamantes oro anillos", "dato":"la mayor parte del oro que existe en la Tierra en realidad se formó en violentas colisiones cósmicas de estrellas de neutrones en el espacio exterior."},
    {"clave":"lluvia agua clima llover", "dato":"el inconfundible olor a tierra mojada después de la lluvia tiene un nombre científico oficial: se llama petricor."},
    {"clave":"frio invierno nieve hielo", "dato":"los copos de nieve pueden tardar hasta una hora completa en caer desde las nubes hasta tocar la superficie terrestre debido a su bajísimo peso."},
    {"clave":"calor verano sol playa", "dato":"la arena de las playas de color blanco brillante suele estar compuesta por pequeños restos de corales o conchas molidas durante milenios por el mar."},
    {"clave":"montaña escalar alpinismo", "dato":"las cordilleras más altas del mundo continúan creciendo en la actualidad un par de centímetros cada año por el choque lento de las placas tectónicas."},
    {"clave":"ciudad rascacielos metropoli", "dato":"en la ciudad de tokio se encuentra la intersección peatonal más transitada del planeta, cruzada por más de tres mil personas en un solo cambio de semáforo."},
    {"clave":"colores azul rojo amarillo", "dato":"el color que los ojos humanos pueden detectar con mayor facilidad a la distancia, incluso con poca luz, es una tonalidad exacta de amarillo verdoso."},
    {"clave":"estrellas luna espacio cosmos", "dato":"el universo no es silencioso del todo; ciertos instrumentos científicos pueden traducir la radiación cósmica en frecuencias audibles de sonido estático."},
    {"clave":"musica clasica beethoven mozart", "dato":"las plantas domésticas tienden a crecer con un poco más de vigor e inclinación cuando están expuestas diariamente a ritmos de música clásica suave."},
    {"clave":"dulces gomitas caramelo azucar", "dato":"el algodón de azúcar fue paradójicamente co-inventado y patentado por un médico dentista a finales del siglo diecinueve."}
]

def dato_semantico(texto, dataset):
    p = perfil_ngramas(texto, 3)
    mejor = None
    mejor_s = -1.0
    
    for item in dataset:
        s = coseno_perfiles(p, perfil_ngramas(item["clave"], 3))
        if s > mejor_s:
            mejor_s = s; mejor = item
    if mejor is None:
        return "no tengo un dato relacionado."
    return mejor["dato"]

def extraer_anio(texto):
    m = re.search(r"\b(?:19|20)\d{2}\b", texto)
    if m:
        return int(m.group(0))
    return None

In [48]:
# resumen (word2vec) — version de main_summary.py adaptada a esta libreta
ORACIONES_RESUMEN = 7
TAMANIO_VECTOR = 100
VENTANA = 4
MIN_COUNT = 1
PENALIZACION_REDUNDANCIA = 0.35  # reservado: no se usa todavia (viene asi desde main_summary.py)

# stopwords propias del resumen. Distintas de las de tokenizar() porque aqui si
# queremos filtrar conectores y palabras muy cortas para que Word2Vec no se
# entrene con ruido.
STOPWORDS_RESUMEN = {
    "a", "al", "algo", "ante", "asi", "cada", "como", "con", "contra",
    "cuando", "de", "del", "desde", "donde", "dos", "e", "el", "en",
    "entre", "era", "es", "esa", "ese", "eso", "esta", "este", "esto",
    "estos", "fue", "ha", "hay", "la", "las", "lo", "los", "mas",
    "me", "mi", "mientras", "muy", "no", "o", "para", "pero", "por",
    "porque", "que", "se", "ser", "si", "sin", "sobre", "son", "su",
    "sus", "tambien", "un", "una", "uno", "y", "ya"
}

def separar_oraciones(texto):
    texto = re.sub(r"\s+", " ", texto.strip())
    oraciones = re.split(r"(?<=[.!?])\s+", texto)
    return [oracion.strip() for oracion in oraciones if oracion.strip()]

def tokenizar_resumen(texto):
    # Equivalente al tokenizar() de main_summary.py. Se renombra para no pisar
    # la tokenizar() global de esta libreta (esa no quita stopwords ni filtra
    # por longitud, y la usan es_negacion/es_afirmacion y el conteo de palabras
    # del resumen final).
    texto = texto.lower()
    tokens = re.findall(r"\b[a-záéíóúñü]+\b", texto)
    return [token for token in tokens if token not in STOPWORDS_RESUMEN and len(token) > 2]

def entrenar_word2vec(oraciones_tokenizadas):
    modelo = Word2Vec(
        sentences=oraciones_tokenizadas,
        vector_size=TAMANIO_VECTOR,
        window=VENTANA,
        min_count=MIN_COUNT,
        workers=1,
        sg=1,
        epochs=200,
        seed=42,
    )
    return modelo

def vector_oracion(tokens, modelo):
    vectores = [modelo.wv[token] for token in tokens if token in modelo.wv]
    if not vectores:
        return np.zeros(TAMANIO_VECTOR)
    return np.mean(vectores, axis=0)

def seleccionar_oraciones(vectores_oraciones, vector_documento, cantidad):
    # Usa coseno(), ya definido arriba (es la misma formula que
    # similitud_coseno() de main_summary.py), para no duplicar la metrica.
    puntajes = [
        (indice, coseno(vectores_oraciones[indice], vector_documento))
        for indice in range(len(vectores_oraciones))
    ]
    oraciones_por_puntaje = sorted(puntajes, key=lambda x: x[1], reverse=True)
    oraciones_por_indice = sorted(oraciones_por_puntaje[:cantidad], key=lambda x: x[0])
    return [indice for indice, _puntaje in oraciones_por_indice]

def resumir(texto, cantidad_oraciones=ORACIONES_RESUMEN):
    oraciones = separar_oraciones(texto)
    oraciones_tokenizadas = [tokenizar_resumen(o) for o in oraciones]

    modelo = entrenar_word2vec(oraciones_tokenizadas)

    vectores_oraciones = np.array([
        vector_oracion(tokens, modelo)
        for tokens in oraciones_tokenizadas
    ])

    vector_documento = np.mean(vectores_oraciones, axis=0)
    indices_resumen = seleccionar_oraciones(
        vectores_oraciones,
        vector_documento,
        cantidad_oraciones
    )

    resumen = " ".join(oraciones[indice] for indice in indices_resumen)
    return resumen, indices_resumen

In [49]:
# saludo
def franja_horaria():
    h = datetime.now().hour
    if h < 12: return "Buenos dias"
    if h < 19: return "Buenas tardes"
    return "Buenas noches"

def clima_actual(lat=19.505579, lon=-99.146089):
    try:
        import urllib.request, json
        url = ("https://api.open-meteo.com/v1/forecast?latitude="
               + str(lat) + "&longitude=" + str(lon) + "&current=temperature_2m")
        with urllib.request.urlopen(url, timeout=4) as r:
            d = json.loads(r.read())
        return "hay " + str(d["current"]["temperature_2m"]) + " grados"
    except Exception:
        return None

def construir_saludo(nombre):
    s = franja_horaria()
    c = clima_actual()
    if c:
        return s + ", " + nombre + ". Ahora mismo " + c + " por aqui. Cuentame, como estuvo tu dia?"
    return s + ", " + nombre + ". Cuentame, como estuvo tu dia?"

In [50]:
# stt (whisper)
_WHISPER = {}
def escuchar(segundos=6, fs=16000):

    import sounddevice as sd
    from faster_whisper import WhisperModel
    
    if "m" not in _WHISPER:
        _WHISPER["m"] = WhisperModel("small", device="cpu", compute_type="int8")
    log("...escuchando " + str(segundos) + "s...")
    audio = sd.rec(int(segundos * fs), samplerate=fs, channels=1, dtype="float32")
    sd.wait()
    audio = audio.reshape(-1)
    segs, _ = _WHISPER["m"].transcribe(audio, language="es", vad_filter=True)
    texto = ""
    for s in segs:
        texto += s.text
    return texto.strip()

In [51]:
# tts (piper)
VOZ_PIPER = "es_MX-claude-high"
DIR_VOCES = Path("voces_piper")
DIR_VOCES.mkdir(exist_ok=True)

_PIPER = {}
def cargar_piper(voz=VOZ_PIPER):
    from piper import PiperVoice
    from piper.download_voices import download_voice
    if "v" not in _PIPER:
        modelo = DIR_VOCES / (voz + ".onnx")
        if not modelo.exists():
            log("Descargando voz '" + voz + "'...", "det")
            download_voice(voz, DIR_VOCES)
        _PIPER["v"] = PiperVoice.load(str(modelo))
    return _PIPER["v"]

def hablar(texto):
    log(texto, "alexa")
    import sounddevice as sd
    voz = cargar_piper()
    trozos = [chunk.audio_int16_array for chunk in voz.synthesize(texto)]
    audio = np.concatenate(trozos)
    sd.play(audio, voz.config.sample_rate)
    sd.wait()

In [52]:
# respuestas
def respuesta_sentimiento(sent, usuario):
    if sent == "positivo":
        return "Me alegra mucho, " + usuario + "! Suena a un buen dia."
    if sent == "negativo":
        return "Lamento que no haya sido un buen dia, " + usuario + ". Espero que mejore pronto."
    return "Gracias por contarme, " + usuario + "."

def es_negacion(texto):
    t = A_minusculas(texto)
    for w in ["no gracias", "no, gracias", "ya no", "adios", "salir", "terminar", "no"]:
        if w in t:
            return True
    tokens = tokenizar(texto)
    return ("no" in tokens) and ("si" not in tokens)

def es_afirmacion(texto):
    tokens = tokenizar(texto)
    for w in ["si", "claro", "va", "dale", "adelante", "bueno", "obvio", "sí"]:
        if w in tokens:
            return True
    return False

In [53]:
# cuento
CUENTO = """
El viejo faro se alzaba al final del acantilado desde hacia mas de cien años.
Marina llego al pueblo buscando silencio para terminar de escribir su novela.
La gente del lugar le advirtio que nadie vivia cerca del faro desde hacia decadas.
Decian que las noches de tormenta una luz se encendia sola en lo alto de la torre.
Marina no creia en fantasmas, asi que alquilo la casa mas cercana al acantilado.
La primera semana fue tranquila y avanzo mucho en su historia.
Una noche de viento fuerte, Marina vio la luz del faro encenderse en la oscuridad.
Curiosa y un poco asustada, decidio subir por el sendero hasta la torre.
Dentro encontro a un anciano que cuidaba la lampara con manos temblorosas.
El hombre le conto que habia sido el ultimo farero antes de que cerraran el faro.
Cada tormenta regresaba para encender la luz, porque temia que un barco se perdiera.
Marina comprendio que aquella costumbre era su forma de seguir siendo util.
Conmovida, le ofrecio ayuda para mantener el faro encendido durante el invierno.
El anciano sonrio por primera vez en muchos años y acepto la compañia.
Desde entonces, cada noche de tormenta, dos luces brillaban en el acantilado.
Una era la del faro y la otra la de la ventana donde Marina escribia.
El pueblo dejo de hablar de fantasmas y empezo a hablar de la escritora y el farero.
La novela de Marina se publico al año siguiente y se titulo precisamente El faro.
En la dedicatoria escribio que la soledad a veces solo necesita una luz al lado.
El anciano murio en paz aquel verano, pero la luz del faro nunca volvio a apagarse.
Marina se quedo a vivir en el pueblo y se encargo de cuidar la lampara cada tormenta.
"""

In [54]:
# maquina de estados
_ctx = {}
def correr_alexa():
    estado = "ESPERANDO_SALUDO"
    usuario = None
    while True:
        log(estado, "estado")
        if estado == "ESPERANDO_SALUDO":
            dicho = escuchar()
            if dicho.strip() == "":
                continue
            nombre, score = reconocer_integrante(dicho)
            log("saludo='" + dicho + "'  match=" + str(nombre) + "  score=" + str(round(score, 2)), "det")
            if nombre is None:
                continue
            usuario = nombre
            hablar(construir_saludo(usuario))
            estado = "DESCRIBIR_DIA"

        elif estado == "DESCRIBIR_DIA":
            desc = escuchar()
            sent = analizar_sentimiento(desc)
            log("sentimiento=" + sent, "det")
            hablar(respuesta_sentimiento(sent, usuario))
            estado = "PEDIR_DATOS"

        elif estado == "PEDIR_DATOS":
            hablar("Dime un año, tu pasatiempo favorito y algo que te guste, todo junto.")
            datos = escuchar()
            _ctx["anio"] = extraer_anio(datos)
            _ctx["datos"] = datos
            log("año=" + str(_ctx["anio"]), "det")
            estado = "DATO_CURIOSO"

        elif estado == "DATO_CURIOSO":
            anio = _ctx.get("anio")
            datos = _ctx.get("datos", "")
            if anio is not None:
                hablar("Sobre " + str(anio) + ": " + dato_anio(anio))
            hablar("Sobre tu pasatiempo: " + dato_semantico(datos, DATOS_PASATIEMPO))
            hablar("Y algo curioso: " + dato_semantico(datos, DATOS_GUSTO))
            estado = "OFRECER_RESUMEN"

        elif estado == "OFRECER_RESUMEN":
            hablar("Quieres escuchar el resumen del cuento? Si no, di 'No, gracias' para terminar.")
            r = escuchar()
            if es_negacion(r):
                hablar("De acuerdo, " + usuario + ". Hasta luego!")
                estado = "ESPERANDO_SALUDO"; usuario = None
            elif es_afirmacion(r):
                estado = "REPRODUCIR_RESUMEN"
            else:
                hablar("No te entendi, repite por favor.")

        elif estado == "REPRODUCIR_RESUMEN":
            resumen, indices = resumir(CUENTO, 4)
            log("oraciones elegidas: " + str(indices), "det")
            log("resumen de " + str(len(tokenizar(resumen))) + " palabras", "det")
            hablar(resumen)
            hablar("Eso es todo, " + usuario + ". Hasta luego!")
            estado = "ESPERANDO_SALUDO"; usuario = None

In [55]:
# ejecutar
correr_alexa()

>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='¿Dame con el viso completo? ¿Qué onda Alexa? Ya. Ya se murió.'  match=None  score=0.5
>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='¿Tienes tu culpa, dice? ¿Qué onda, Alexa?'  match=None  score=0.38
>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='Si, no utilizadame a todos los Google supportos. ¿Es asesina? Es que hay detectadores.'  match=None  score=0.27
>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='¿Cuál van a entregar? Alexa, ¿qué onda? Que muchas veces.'  match=None  score=0.49
>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='es que la gente va a salir por van a entregarse'  match=None  score=0.42
>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='¡Qué analexa!'  match=None  score=0.48
>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='¿Qué ond

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3884.37it/s]


  [DETEC]  sentimiento=positivo
  [ALEXA]  Me alegra mucho, Fer! Suena a un buen dia.
>>ESTADO  PEDIR_DATOS
  [ALEXA]  Dime un año, tu pasatiempo favorito y algo que te guste, todo junto.
          ...escuchando 6s...
  [DETEC]  año=2020
>>ESTADO  DATO_CURIOSO
  [ALEXA]  Sobre 2020: en 2020 el mundo vivió una pandemia global que popularizó el trabajo remoto.
  [ALEXA]  Sobre tu pasatiempo: el basquetbol se inventó en invierno con cestas de duraznos reales, por lo que inicialmente tenían que empujar el balón con un palo tras cada punto.
  [ALEXA]  Y algo curioso: las plantas en un bosque están interconectadas por redes de hongos subterráneas a través de las cuales se envían nutrientes y alertas de peligro.
>>ESTADO  OFRECER_RESUMEN
  [ALEXA]  Quieres escuchar el resumen del cuento? Si no, di 'No, gracias' para terminar.
          ...escuchando 6s...
  [ALEXA]  De acuerdo, Fer. Hasta luego!
>>ESTADO  ESPERANDO_SALUDO
          ...escuchando 6s...
  [DETEC]  saludo='Ahora ponrelo aquí est

KeyboardInterrupt: 